> DIFF (2026-03-19): 新規作成 `[4-7]submit-notebook-v3.ipynb`（元: `[4-6]submit-notebook-v3.ipynb` / モデル: `[2-7]dpc-starter-train-v3.ipynb`）

# [4-7] submit-notebook-v3

**差分メモ（2026-03-19）**

- 対象: `[4-7]submit-notebook-v3.ipynb`
- 元: `[4-6]submit-notebook-v3.ipynb`
- 種別: 枝分かれ
- 変更点:
  - 学習済みモデルの参照先を `notebooks/002/[2-7]dpc-starter-train-v3.ipynb` の出力 (`fulltrain_byt5-small_multitask`) に更新
  - 前処理/後処理は `notebooks/002/[2-7]dpc-starter-train-v3.ipynb` と合わせる（同一ロジック前提）
  - `[4-6]submit-notebook-v3.ipynb` の scoring error 対策（提出直前バリデーション + 空文字対策）は踏襲
- 変更理由/仮説:
  - curated train v002-4 で学習したモデルを、その学習時と同じ正規化ルールで安全に submit したい

## Model path
`MODEL_DIR` は差し替えやすいように集約しています。Kaggle Dataset / Model として学習済みモデルをアタッチし、必要に応じて `MODEL_DIR` を変更してください。



In [1]:
# DIFF (2026-03-19): 元 `[4-6]submit-notebook-v3.ipynb` から変更なし（import セル）
import os
import math
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


'false'

In [2]:
# DIFF (2026-03-19): `CFG.model_dir` の既定値を `[2-7]dpc-starter-train-v3.ipynb` の出力に更新
@dataclass
class CFG:
    # Competition data path (Kaggle may mount it in either location)
    input_dirs: tuple[str, ...] = (
        "/kaggle/input/competitions/deep-past-initiative-machine-translation",
        "/kaggle/input/deep-past-initiative-machine-translation",
    )

    # >>> Replace this later (intentionally centralized)
    # Example (dataset): /kaggle/input/<your-dataset>/<your-model-folder>
    # Example (model):   /kaggle/input/models/<owner>/<model>/pytorch/<variant>/<version>
    model_dir: str = os.environ.get("MODEL_DIR", "/kaggle/input/notebooks/tatsuokoshida/2-7-dpc-starter-train-v3/byt5-akkadian-model/fulltrain_byt5-small_multitask")

    output_dir: str = "/kaggle/working"

    # Tokenization / generation (match train notebook defaults)
    max_input_length: int = 512
    max_new_tokens: int = 512

    # Inference
    batch_size: int = 8
    num_workers: int = 2
    num_beams: int = 4
    early_stopping: bool = True

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


cfg = CFG()
print("PyTorch:", torch.__version__)
print("Device :", cfg.device)


PyTorch: 2.9.0+cu126
Device : cuda


In [3]:
# DIFF (2026-03-19): 前処理/後処理は `[2-7]dpc-starter-train-v3.ipynb` と同一ロジック前提（submit 側は build_inputs を利用）
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
from typing import List


PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

# --- Pre/Post processing (aligned to notebooks/006/lb-35-9-ensembling-post-processing-baseline.ipynb) ---
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


# Unicode fraction map (all single-character Unicode vulgar fractions)
_UNICODE_FRAC_MAP = {
    (1, 2): "½", (1, 3): "⅓", (2, 3): "⅔",
    (1, 4): "¼", (3, 4): "¾",
    (1, 5): "⅕", (2, 5): "⅖", (3, 5): "⅗", (4, 5): "⅘",
    (1, 6): "⅙", (5, 6): "⅚",
    (1, 7): "⅐",
    (1, 8): "⅛", (3, 8): "⅜", (5, 8): "⅝", (7, 8): "⅞",
    (1, 9): "⅑",
    (1, 10): "⅒",
}
def _trunc_float(x: float, places: int = 4) -> float:
    factor = 10 ** places
    if x >= 0:
        return math.floor(x * factor) / factor
    return -math.floor(-x * factor) / factor

def _fmt_trunc(x: float, places: int = 4) -> str:
    return f"{_trunc_float(x, places):.{places}f}".rstrip("0").rstrip(".")

# Map the 4-decimal *truncated* representation to a Unicode fraction.
# (Host example: 1.333330000... -> 1.3333; 0.1666 -> ⅙)
_FRAC_DECIMAL_MAP_4 = {}
for (n, d), ch in _UNICODE_FRAC_MAP.items():
    _FRAC_DECIMAL_MAP_4[_fmt_trunc(n / d, 4)] = ch

# Decimals (incl. long floats)
_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d+)(?![\w/])")
_LONG_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d{5,})(?![\w/])")

# General fractions like "1/2" or mixed fractions like "2 1/2".
_MIXED_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s+(\d+)\s*/\s*(\d+)(?![\w/])")
_SIMPLE_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s*/\s*(\d+)(?![\w/])")

def _mixed_frac_repl(m: re.Match) -> str:
    ip = int(m.group(1))
    num = int(m.group(2))
    den = int(m.group(3))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return f"{ip} {ch}" if ip != 0 else ch
    return m.group(0)

def _simple_frac_repl(m: re.Match) -> str:
    num = int(m.group(1))
    den = int(m.group(2))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return ch
    return m.group(0)

def _canon_decimal_str(s: str) -> str:
    # Keep x.5 (e.g., 2.5) as-is by request.
    if re.fullmatch(r"[1-9]\d*\.5", s):
        return s

    m = re.fullmatch(r"(\d+)\.(\d+)", s)
    if not m:
        return s

    ip = int(m.group(1))
    frac_digits = m.group(2)

    # Truncate fractional part to 4 digits (no rounding), pad with zeros for lookup.
    frac4 = (frac_digits + "0000")[:4]
    frac_key = ("0." + frac4).rstrip("0").rstrip(".")
    ch = _FRAC_DECIMAL_MAP_4.get(frac_key)
    if ch and frac_key != "0":
        if ip == 0:
            return ch
        return f"{ip} {ch}"

    # Long-float shortening: truncate to 4 digits after the decimal.
    if len(frac_digits) > 4:
        return f"{ip}.{frac4}".rstrip("0").rstrip(".")

    return s


_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_TAG_GAP_RE    = re.compile(r"<\s*gap\s*>", re.I)
_BARE_BIGGAP   = re.compile(r"\bbig[\s_\-]*gap\b", re.I)
_ELLIPSIS_RE   = re.compile(r"(?:\.{3,}|…+|\[\.+\])")
_BRACKET_X_RE  = re.compile(r"(\[\s*x\s*\]|\(\s*x\s*\))", re.I)
_XTOKEN_RUN_RE = re.compile(r"\bx(?:\s+x)+\b", re.I)
_XRUN_RE       = re.compile(r"(?<!\w)x{2,}(?!\w)", re.I)
_XTOK_RE       = re.compile(r"(?<!\w)x(?!\w)", re.I)
_WS_RE         = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)


_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")


class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)

        # Uppercase determinatives are unwrapped, lowercase ones are converted to braces.
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)

        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        ser = ser.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        ser = ser.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_QUOTES_RE = re.compile("[\u201c\u201d\u2018\u2019]")

_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}

_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")

_FORBIDDEN_TRANS = str.maketrans("", "", '()——<>⌈⌋⌊[]+ʾ;')

_COMMODITY_RE = re.compile(r'-(gold|tax|textiles)\b')
_COMMODITY_REPL = {
    "gold": "pašallum gold",
    "tax": "šadduātum tax",
    "textiles": "kutānum textiles",
}

def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]


_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅔ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

_ALT_SLASH_PAIR_RE = re.compile(r"\b([^\W\d_]+)\s*/\s*([^\W\d_]+)\b")
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)

        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)

        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)

        s = s.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        s = s.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        s = s.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)

        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)

        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        # Keep the left alternative (e.g., "you / she" -> "you")
        s = s.str.replace(_ALT_SLASH_PAIR_RE, r"\1", regex=True)

        # Remove only curly quotes; keep straight quotes and apostrophes.
        s = s.str.replace(_CURLY_QUOTES_RE, "", regex=True)

        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)

        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)

        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)

        s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()

        return s.tolist()

_preprocessor = OptimizedPreprocessor()
_postprocessor = VectorizedPostprocessor()

def resolve_input_dir(input_dirs: tuple[str, ...]) -> str:
    for d in input_dirs:
        if Path(d).exists():
            return d
    raise FileNotFoundError(
        "Competition input directory not found. Tried: " + ", ".join(input_dirs)
    )


def build_inputs(test_df: pd.DataFrame) -> tuple[list[str], list[str], list[str]]:
    if "id" not in test_df.columns:
        raise ValueError(f"test.csv must contain 'id'. columns={list(test_df.columns)}")
    ids = test_df["id"].astype(str).tolist()

    if "task" in test_df.columns:
        tasks = test_df["task"].astype(str).tolist()
    else:
        tasks = ["akk2en"] * len(test_df)

    if "source" in test_df.columns:
        sources = test_df["source"].astype(str).tolist()
    elif "transliteration" in test_df.columns:
        sources = test_df["transliteration"].astype(str).tolist()
    else:
        raise ValueError(
            "test.csv must contain either 'source' or 'transliteration'. "
            f"columns={list(test_df.columns)}"
        )

    inputs: list[str] = [""] * len(sources)

    idx_akk2en = [i for i, t in enumerate(tasks) if t == "akk2en"]
    idx_en2akk = [i for i, t in enumerate(tasks) if t == "en2akk"]

    src_akk = [sources[i] for i in idx_akk2en]
    src_en = [sources[i] for i in idx_en2akk]

    src_akk_pp = _preprocessor.preprocess_batch(src_akk) if src_akk else []
    src_en_pp = _postprocessor.postprocess_batch(src_en) if src_en else []

    for i, s_norm in zip(idx_akk2en, src_akk_pp):
        inputs[i] = PREFIX_AKK_EN + s_norm
    for i, s_norm in zip(idx_en2akk, src_en_pp):
        inputs[i] = PREFIX_EN_AKK + s_norm

    # Keep strict: unknown tasks are a hard error.
    unknown = sorted(set(tasks) - {"akk2en", "en2akk"})
    if unknown:
        raise ValueError(f"Unknown task(s): {unknown}")

    return ids, tasks, inputs


In [4]:
# DIFF (2026-03-19): `resolve_model_dir` の既定候補を `[2-7]` の出力フォルダ名に更新
def resolve_model_dir(model_dir: str) -> str:
    candidates = [
        model_dir,
        # Useful when you ran training in the same notebook session and kept outputs in /kaggle/working
        "/kaggle/working/byt5-akkadian-model/fulltrain_byt5-small_multitask",
    ]
    for d in candidates:
        if Path(d).exists():
            return d
    raise FileNotFoundError(
        "Model directory not found. Set MODEL_DIR to a mounted path containing the saved model. "
        f"Tried: {candidates}"
    )


input_dir = resolve_input_dir(cfg.input_dirs)
test_path = f"{input_dir}/test.csv"

print("Test:", test_path)

test_df = pd.read_csv(test_path)
print("Test rows:", len(test_df))
print("Columns:", list(test_df.columns))

model_dir = resolve_model_dir(cfg.model_dir)
print("Model:", model_dir)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
model.to(cfg.device)
model.eval()

ids, tasks, inputs = build_inputs(test_df)
print("Prepared inputs:", len(inputs))


Test: /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
Test rows: 4
Columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']
Model: /kaggle/input/notebooks/tatsuokoshida/2-3-dpc-starter-train-v3/byt5-akkadian-model/fulltrain_byt5-small_multitask


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Prepared inputs: 4


In [5]:
# DIFF (2026-03-19): 元 `[4-6]submit-notebook-v3.ipynb` から変更なし（推論 + postprocess）
class TextDataset(Dataset):
    def __init__(self, texts: list[str], tasks: list[str]):
        if len(texts) != len(tasks):
            raise ValueError(f"len(texts)={len(texts)} != len(tasks)={len(tasks)}")
        self.texts = texts
        self.tasks = tasks

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        return {"text": self.texts[idx], "task": self.tasks[idx]}


def collate_fn(batch: list[dict]):
    texts = [x["text"] for x in batch]
    tasks = [x["task"] for x in batch]
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.max_input_length,
    )
    enc["tasks"] = tasks
    return enc


ds = TextDataset(inputs, tasks)
dl = DataLoader(
    ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=(cfg.device.type == "cuda"),
)


def _move_to_device(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out


def postprocess_batch(out_texts: list[str], tasks: list[str]) -> list[str]:
    if len(out_texts) != len(tasks):
        raise ValueError(f"len(out_texts)={len(out_texts)} != len(tasks)={len(tasks)}")
    out: list[str] = [""] * len(out_texts)

    idx_akk2en = [i for i, t in enumerate(tasks) if t == "akk2en"]
    idx_en2akk = [i for i, t in enumerate(tasks) if t == "en2akk"]

    if idx_akk2en:
        texts = [out_texts[i] for i in idx_akk2en]
        pp = _postprocessor.postprocess_batch(texts)
        for i, t in zip(idx_akk2en, pp):
            out[i] = t
    if idx_en2akk:
        texts = [out_texts[i] for i in idx_en2akk]
        pp = _preprocessor.preprocess_batch(texts)
        for i, t in zip(idx_en2akk, pp):
            out[i] = t

    unknown = sorted(set(tasks) - {"akk2en", "en2akk"})
    if unknown:
        raise ValueError(f"Unknown task(s): {unknown}")
    return out


preds: list[str] = []
with torch.inference_mode():
    for batch in tqdm(dl, desc="Generating"):
        batch = _move_to_device(batch, cfg.device)
        gen_ids = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch.get("attention_mask"),
            num_beams=cfg.num_beams,
            early_stopping=cfg.early_stopping,
            max_new_tokens=cfg.max_new_tokens,
        )
        out_texts = tokenizer.batch_decode(
            gen_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        preds.extend(postprocess_batch(out_texts, batch["tasks"]))

assert len(preds) == len(ids)
print("Preds:", len(preds))
print("Example:", preds[0][:200] if preds else "<empty>")


Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Preds: 4
Example: From Kanesh to our messenger, our messenger, our messenger, our messenger and our messenger: The tablet concerning the City will come.


In [6]:
# DIFF (2026-03-19): scoring error 対策（空文字→<gap> + validate_submission）は `[4-6]` を踏襲
sub_df = pd.DataFrame({"id": ids, "translation": preds})
sub_df["translation"] = sub_df["translation"].fillna("").astype(str)
sub_df.loc[sub_df["translation"].str.strip() == "", "translation"] = "<gap>"

def validate_submission(df: pd.DataFrame, expected_rows: int):
    # Kaggle の format error を避けるため、提出直前に最低限の形を検査する。
    expected_columns = ["id", "translation"]
    if list(df.columns) != expected_columns:
        raise ValueError(f"Submission columns must be {expected_columns}. got={list(df.columns)}")
    if len(df) != expected_rows:
        raise ValueError(f"Submission rows mismatch. expected={expected_rows}, got={len(df)}")
    if df["id"].isna().any():
        raise ValueError("Submission contains missing id values.")
    if df["id"].duplicated().any():
        dup_ids = df.loc[df["id"].duplicated(), "id"].astype(str).head(5).tolist()
        raise ValueError(f"Submission contains duplicated id values. examples={dup_ids}")
    empty_mask = df["translation"].fillna("").astype(str).str.strip() == ""
    if empty_mask.any():
        bad_ids = df.loc[empty_mask, "id"].astype(str).head(5).tolist()
        raise ValueError(f"Submission contains empty translations. examples={bad_ids}")

validate_submission(sub_df, expected_rows=len(ids))

sub_path = Path(cfg.output_dir) / "submission.csv"
sub_df.to_csv(sub_path, index=False)

print("Submission preview:")
display(sub_df.head())
print("Saved to:", str(sub_path))


Submission preview:


,id,translation
0,0,"From Kanesh to our messenger, our messenger, o..."
1,1,"As for the tablet in the City, when the very d..."
2,2,Instead of our messenger you have written me a...
3,3,"Indeed, the messenger of our messenger and the..."


Saved to: /kaggle/working/submission.csv
